# Burnt-area mapping of the 2021 Manavgat & Marmaris wildfires — end-to-end demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/html37/ehb466-burnt-area-tr/blob/main/notebooks/00_demo_end_to_end.ipynb)

EHB 466E term project (İTÜ, Fall 2026). This notebook runs the whole pipeline on the **Colab free tier** (CPU is enough):

1. Sentinel-2 L2A pre/post scenes from Microsoft Planetary Computer (tile mosaicking, SCL masks, 20 m grid, FLOGA band order)
2. **dNBR** benchmark (USGS threshold 0.10 and the threshold tuned on the Greek FLOGA validation split)
3. **Our method**: LightGBM pixel classifier trained only on Greek FLOGA data (no Turkish training data), downloaded pre-trained
4. **BAM-CD** benchmark (Sdraka et al., 2024) with the official pretrained weights (1 GB download; CPU inference uses non-overlapping windows)
5. Metrics vs the EFFIS reference perimeters and error maps

Expected runtime on Colab CPU: see `docs/demo_runtime.md` (target ≤ 30 min). Runtime → *Run all*.

In [ ]:
import os, sys, time
T0 = time.time()
REPO = "https://github.com/html37/ehb466-burnt-area-tr.git"
if "google.colab" in sys.modules or not os.path.exists("src/burnt"):
    if not os.path.exists("ehb466-burnt-area-tr"):
        !git clone -q --depth 1 {REPO}
    os.chdir("ehb466-burnt-area-tr")
print(os.getcwd())

In [ ]:
# packages missing on Colab (numpy/pandas/torch/lightgbm/sklearn/matplotlib are preinstalled)
!pip -q install pystac-client planetary-computer rioxarray segmentation-models-pytorch pretrainedmodels

### Run the demo
`EVENTS` can be `['Manavgat']` (fastest) or `['Manavgat', 'Marmaris']`; `--bamcd off` skips the deep-learning benchmark.

In [ ]:
EVENTS = "Manavgat Marmaris"
!python scripts/demo_run.py --events {EVENTS} --bamcd auto

### Results

In [ ]:
import pandas as pd, json
from IPython.display import Image, display
display(pd.read_csv("outputs/demo/demo_metrics.csv"))
for e in EVENTS.split():
    display(Image(f"outputs/demo/demo_{e}.png"))
print(json.load(open("outputs/demo/demo_runtime.json")))
print(f"notebook wall time: {(time.time() - T0) / 60:.1f} min")

### Notes
* Nothing in this demo is tuned on the Turkish data: the LightGBM model, its decision threshold and the dNBR threshold were fixed on the Greek FLOGA dataset (train/validation splits).
* Full experiments (5 seeds, bootstrap CIs, ablation, error analysis, cost table): `python run_all.py` (see README).
* Data: Copernicus Sentinel-2 (Planetary Computer), EFFIS perimeters (© European Union), FLOGA (CC BY 4.0), BAM-CD code (MIT).